# Vestibular Schwannoma Data Ingestion & Organization

1. Load manifest.csv and build absolute paths to imaging files.
2. Validate paths exist on disk, report missing ones.
3. Merge MRI rows with their corresponding mask rows (keep all masks, flag multiples).
4. Extract Patient_ID / Scan_Order / Days from `patient_mri_days`.
5. Filter to desired series tags and remove excluded study labels.
6. Copy NIfTI files into an organized folder structure.
7. Rename columns and export cleaned metadata CSV for downstream processing.

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import shutil

import pandas as pd
from tqdm.auto import tqdm

In [ ]:
# ── Configuration ──
MANIFEST_CSV = '/standard/gam_ai_group/data_raw/uva_vs_notreat/data_freezes/uva_vs_raw_2026_03_04/manifest.csv'
ROOT_DIR = Path('/standard/gam_ai_group/data_raw/uva_vs_notreat/data_freezes/uva_vs_raw_2026_03_04')

# Series tags to keep
KEEP_SERIES_TAGS = ['t2_thin', 't1+_thin', 't1+_thick_cor', 't1+_thick_ax', 't2_thick']

# Study labels that MUST be present
REQUIRE_STUDY_LABELS = ['qc1']

# Study labels to exclude
EXCLUDE_STUDY_LABELS = ['not_vs', 'non_contrast']

# Output directory for organized files
BASE_OUTPUT_DIR = '/standard/gam_ai_group/dataset_curated/uva_vs_notreat'

## 1. Load Manifest

In [ ]:
manifest_df = pd.read_csv(MANIFEST_CSV)
print(f'Total manifest rows: {len(manifest_df)}')
print(f'Columns: {manifest_df.columns.tolist()}')
print(f'\nfile_type counts:')
print(manifest_df['file_type'].value_counts())
print(f'\nseries_tag counts:')
print(manifest_df['series_tag'].value_counts())

## 2. Split MRI / Mask

In [ ]:
mri_df = manifest_df[manifest_df['file_type'] == 'MRI'].copy()
mask_df = manifest_df[manifest_df['file_type'] == 'mask'].copy()

print(f'MRI rows: {len(mri_df)}')
print(f'Mask rows: {len(mask_df)}')

## 3. Build Absolute Paths

In [ ]:
def build_abs_path(file_path):
    """Build absolute path from the manifest's relative file_path."""
    if pd.isna(file_path) or not str(file_path).strip():
        return None
    return str(ROOT_DIR / str(file_path).strip())


mri_df['abs_path'] = mri_df['file_path'].apply(build_abs_path)
mask_df['abs_path'] = mask_df['file_path'].apply(build_abs_path)

## 4. Validate Paths

In [ ]:
import difflib


def validate_paths(df, path_col='abs_path'):
    """Check existence on disk. Returns (valid_df, missing_df)."""
    df['exists'] = df[path_col].apply(
        lambda p: Path(p).exists() if pd.notna(p) and p else False
    )
    valid_df = df[df['exists']].drop(columns=['exists']).copy()
    missing_df = df[~df['exists']].drop(columns=['exists']).copy()
    print(f'  Valid: {len(valid_df)}, Missing: {len(missing_df)}')
    return valid_df, missing_df


def resolve_mask_path(abs_path, mask_name):
    """
    If abs_path doesn't exist, look in its parent directory for a file
    that fuzzy-matches mask_name (handles naming mismatches like
    'LVS _uvauser2.nii.gz' vs 'L_VS__uvauser2_.nii.gz').
    """
    if pd.isna(abs_path) or not abs_path:
        return None
    if os.path.exists(abs_path):
        return abs_path

    parent = os.path.dirname(abs_path)
    if not os.path.isdir(parent):
        return None

    # List only .nii/.nii.gz files (skip .dcm and other files)
    candidates = [f for f in os.listdir(parent)
                  if (f.endswith('.nii') or f.endswith('.nii.gz'))
                  and f != 'image.nii.gz']

    if not candidates:
        return None

    # Exact match on mask_name
    if isinstance(mask_name, str) and mask_name in candidates:
        return os.path.join(parent, mask_name)

    # Fuzzy match
    target = os.path.basename(abs_path)
    matches = difflib.get_close_matches(target, candidates, n=1, cutoff=0.4)
    if matches:
        return os.path.join(parent, matches[0])

    # If only one NIfTI candidate (besides image.nii.gz), use it
    if len(candidates) == 1:
        return os.path.join(parent, candidates[0])

    return None


print('=== MRI paths ===')
mri_clean, mri_errors = validate_paths(mri_df)

print('\n=== Mask paths (initial) ===')
mask_valid, mask_missing = validate_paths(mask_df)

# Attempt to resolve missing mask paths
if not mask_missing.empty:
    print(f'\nAttempting to resolve {len(mask_missing)} missing mask paths...')
    mask_missing['resolved_path'] = mask_missing.apply(
        lambda row: resolve_mask_path(row['abs_path'], row.get('mask_name')), axis=1
    )
    recovered = mask_missing[mask_missing['resolved_path'].notna()].copy()
    recovered['abs_path'] = recovered['resolved_path']
    recovered = recovered.drop(columns=['resolved_path'])

    still_missing = mask_missing[mask_missing['resolved_path'].isna()].drop(columns=['resolved_path'])

    mask_clean = pd.concat([mask_valid, recovered], ignore_index=True)
    mask_errors = still_missing
    print(f'  Recovered: {len(recovered)}, Still missing: {len(still_missing)}')
else:
    mask_clean = mask_valid
    mask_errors = mask_missing

print(f'\nFinal: {len(mri_clean)} valid MRI, {len(mri_errors)} missing MRI')
print(f'Final: {len(mask_clean)} valid masks, {len(mask_errors)} missing masks')

## 5. Save Missing Paths

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d')
if not mri_errors.empty:
    mri_errors.to_csv(f'missing_mri_paths_{timestamp}.csv', index=False)
    print(f'Saved {len(mri_errors)} missing MRI paths')
if not mask_errors.empty:
    mask_errors.to_csv(f'missing_mask_paths_{timestamp}.csv', index=False)
    print(f'Saved {len(mask_errors)} missing mask paths')
if mri_errors.empty and mask_errors.empty:
    print('No missing paths.')

## 6. Merge MRI + Masks

Left join MRI onto masks via `(carina_accession_number, series_index)`.  
All mask rows are kept — series with multiple masks are flagged via `multiple_masks`.

In [ ]:
import re

join_cols = ['carina_accession_number', 'series_index']


def mask_priority(row):
    """
    Lower score = higher priority.
    1. admin with qc1 in study_labels
    2. uvauser by number ascending (user1 before user2, etc.)
    3. Everything else (AI, unknown)
    """
    user = str(row.get('mask_user', '')).strip().lower()
    labels = str(row.get('study_labels', '')).strip().lower()

    has_qc1 = 'qc1' in labels

    if user == 'admin' and has_qc1:
        return 0

    # Match user<N> or uvauser<N>
    m = re.search(r'(?:uva)?user(\d+)', user)
    if m:
        return 100 + int(m.group(1))

    # AI / everything else
    return 9999


def pick_best_mask(group):
    """Pick the highest-priority mask from a group."""
    if len(group) == 1:
        return group.iloc[0]
    group = group.copy()
    group['_priority'] = group.apply(mask_priority, axis=1)
    best = group.sort_values('_priority').iloc[0]
    return best.drop('_priority')


# Flag series with multiple masks
mask_counts = mask_clean.groupby(join_cols).size().reset_index(name='_mask_count')
mask_clean = mask_clean.merge(mask_counts, on=join_cols, how='left')
mask_clean['multiple_masks'] = mask_clean['_mask_count'] > 1
mask_clean = mask_clean.drop(columns=['_mask_count'])

multi_mask_series = mask_clean[mask_clean['multiple_masks']].groupby(join_cols).ngroups
print(f'Unique series with multiple masks: {multi_mask_series}')

# Build list of all mask files per series
all_masks_per_series = (
    mask_clean
    .groupby(join_cols)['abs_path']
    .apply(lambda paths: '; '.join(sorted(paths.dropna().astype(str))))
    .reset_index()
    .rename(columns={'abs_path': 'all_segmentation_files'})
)

# Pick best mask per series
best_masks = (
    mask_clean
    .groupby(join_cols, group_keys=False)
    .apply(pick_best_mask)
    .reset_index(drop=True)
)

# Attach the all_segmentation_files list
best_masks = best_masks.merge(all_masks_per_series, on=join_cols, how='left')

print(f'Selected {len(best_masks)} best masks from {len(mask_clean)} total mask rows')

# Rename mask columns before merge
best_masks = best_masks.rename(columns={
    'abs_path': 'segmentation_abs_path',
    'file_path': 'segmentation_file_path',
})

mask_merge_cols = join_cols + [
    'segmentation_abs_path', 'segmentation_file_path',
    'mask_name', 'mask_user', 'mask_volume', 'multiple_masks',
    'all_segmentation_files',
]

# Left join: one row per MRI, with best mask
meta_df = mri_clean.merge(
    best_masks[mask_merge_cols],
    on=join_cols,
    how='left',
)

print(f'\nCombined DataFrame: {len(meta_df)} rows')
print(f'  With segmentation: {meta_df["segmentation_abs_path"].notna().sum()}')
print(f'  Without segmentation: {meta_df["segmentation_abs_path"].isna().sum()}')

## 7. Extract Patient_ID / Scan_Order / Days

In [ ]:
meta_df[['Patient_ID', 'Scan_Order', 'Days_Between_Scans']] = (
    meta_df['patient_mri_days'].str.split('_', expand=True)
)
meta_df = meta_df.sort_values(by=['Patient_ID', 'Scan_Order'])

print(f'Unique patients: {meta_df["Patient_ID"].nunique()}')
print(f'\nseries_tag distribution:')
print(meta_df['series_tag'].value_counts())

## 8. Filter series_tag

In [ ]:
meta_df = meta_df[meta_df['series_tag'].isin(KEEP_SERIES_TAGS)]
print(f'After series_tag filter: {len(meta_df)} rows')
print(meta_df['series_tag'].value_counts())

## 9. Filter study_labels

In [ ]:
# Require study labels
for label in REQUIRE_STUDY_LABELS:
    meta_df = meta_df[meta_df['study_labels'].str.contains(label, case=False, na=False)]
print(f'After requiring {REQUIRE_STUDY_LABELS}: {len(meta_df)} rows')

# Exclude study labels
for label in EXCLUDE_STUDY_LABELS:
    meta_df = meta_df[~meta_df['study_labels'].str.contains(label, case=False, na=False)]
print(f'After excluding {EXCLUDE_STUDY_LABELS}: {len(meta_df)} rows')
print(f'\nFiltered series_tag distribution:')
print(meta_df['series_tag'].value_counts())

## 10. Organize NIfTI Files

Copy to `BASE_OUTPUT_DIR/sorted_unprocessed_studies/{series_tag}/{patient_mri_days}/` with prefixed filenames.

In [ ]:
def copy_to_organized_folder(src_path, base_output_dir, series_tag, patient_mri_days):
    """Copy file to organized folder structure with prefixed filename."""
    if not src_path or not os.path.exists(src_path):
        return None

    dest_dir = os.path.join(base_output_dir, 'sorted_unprocessed_studies', str(series_tag), str(patient_mri_days))
    os.makedirs(dest_dir, exist_ok=True)

    original_filename = os.path.basename(src_path)
    prefixed_filename = f'{patient_mri_days}_{series_tag}_{original_filename}'
    dest_path = os.path.join(dest_dir, prefixed_filename)

    shutil.copy2(src_path, dest_path)
    return dest_path


meta_df['image_organized_path'] = None
meta_df['seg_organized_path'] = None

for idx, row in tqdm(meta_df.iterrows(), total=len(meta_df), desc='Organizing NIfTI files'):
    series_tag = row['series_tag']
    pmd = row['patient_mri_days']

    # Copy image
    img_dest = copy_to_organized_folder(row['abs_path'], BASE_OUTPUT_DIR, series_tag, pmd)
    meta_df.at[idx, 'image_organized_path'] = img_dest

    # Copy segmentation if present
    seg_path = row.get('segmentation_abs_path')
    if pd.notna(seg_path):
        seg_dest = copy_to_organized_folder(seg_path, BASE_OUTPUT_DIR, series_tag, pmd)
        meta_df.at[idx, 'seg_organized_path'] = seg_dest

print(f'\nDone. Images copied: {meta_df["image_organized_path"].notna().sum()}')
print(f'Segmentations copied: {meta_df["seg_organized_path"].notna().sum()}')

## 11. Rename Columns & Export

In [ ]:
meta_df = meta_df.rename(columns={
    'series_tag': 'Study Type',
    'image_organized_path': 'image path',
    'seg_organized_path': 'segmentation path',
    'patient_mri_days': 'Patient_MRI_Days Tracker',
})

# Keep only columns needed for downstream processing
export_cols = [
    'carina_accession_number',
    'series_index',
    'Patient_ID',
    'Scan_Order',
    'Days_Between_Scans',
    'Patient_MRI_Days Tracker',
    'Study Type',
    'image path',
    'segmentation path',
    'mask_name',
    'mask_user',
    'mask_volume',
    'multiple_masks',
    'all_segmentation_files',
]
export_cols = [c for c in export_cols if c in meta_df.columns]
meta_df = meta_df[export_cols]

date_str = datetime.now().strftime('%Y-%m-%d')
output_csv = f'meta_df_{date_str}.csv'
meta_df.to_csv(output_csv, index=False)
print(f'Exported columns: {export_cols}')
print(f'Saved {len(meta_df)} rows to {output_csv}')
meta_df.head()